# Week #3(a): Learning Rules from Examples

[Open this notebook in Colab](https://colab.research.google.com/github/mmoffitt/engineering-intelligence/blob/main/notebooks/week03a.ipynb)

> **Important:** Before you do any work, save your own copy to Google Drive.
>
> In Colab: **File → Save a copy in Drive**

In Week 2, you wrote functions that evaluated and transformed image grids. This week, the instructions are no longer given to you directly. You will study example input/output pairs, infer the hidden rule, and write a program that applies it to a new grid.

That is the basic workflow of an [**ARC-AGI task**](https://arcprize.org/).

## Goals

By the end of the core lesson, you will be able to:

- distinguish **training examples** from a **test input**
- record observations before writing code
- infer a transformation from several examples
- break the transformation into a smaller helper function
- test a proposed rule on grids with different dimensions
- explain why passing one example is not enough evidence

The core lesson is designed to be completed first. The stretch laboratory afterward can take you much further.

## Setup

Run the next cell once. It provides grid drawing functions and the **Check Exercise** buttons.

ARC uses integers `0` through `9` to represent colors. The numbers do not have numerical meaning: `2` means red, not "twice as much" as blue.

In [ ]:
#@title Setup utilities (run once)

import copy
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import ListedColormap

GRID_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25"
]
GRID_CMAP = ListedColormap(GRID_COLORS)


def draw_grid(ax, grid, title=""):
    rows = len(grid)
    cols = len(grid[0]) if rows else 0
    ax.imshow(grid, cmap=GRID_CMAP, vmin=0, vmax=9)
    ax.set_title(title)
    ax.set_xticks([x - 0.5 for x in range(1, cols)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, rows)], minor=True)
    ax.grid(which="minor", color="white", linewidth=1)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)


def show_grid(grid, title="Grid"):
    _, ax = plt.subplots(figsize=(3, 3))
    draw_grid(ax, grid, title)
    plt.show()


def show_pair(input_grid, output_grid, label=""):
    _, axes = plt.subplots(1, 2, figsize=(6, 3))
    draw_grid(axes[0], input_grid, "Input")
    draw_grid(axes[1], output_grid, "Output")
    if label:
        plt.suptitle(label)
    plt.tight_layout()
    plt.show()


def show_examples(examples):
    for number, example in enumerate(examples, start=1):
        show_pair(
            example["input"],
            example["output"],
            f"Training example {number}",
        )


def make_check_button(check_fn):
    button = widgets.Button(description="Check Exercise", button_style="info")
    output = widgets.Output()

    def on_click(_):
        output.clear_output()
        with output:
            check_fn()

    button.on_click(on_click)
    display(button, output)


print("Setup complete.")

## Part 0: Dictionaries

A Python **dictionary** keeps related values together and gives each value a name called a **key**.

```python
student = {
    "name": "Ada",
    "grade": 10,
}
```

This dictionary has two keys: `"name"` and `"grade"`. Use square brackets and a key to retrieve its value:

```python
student["name"]   # "Ada"
student["grade"]  # 10
```

Unlike a list, you do not look up these values by position. You look them up by their descriptive keys.

Python dictionaries *can* be changed. In this notebook, however, we will define each dictionary once and then treat it as **read-only**: we will retrieve its values without adding, replacing, or deleting anything.

## Why dictionaries are useful for ARC

Each ARC training example contains two related grids:

- an input grid
- its expected output grid

A dictionary gives those grids clear names:

```python
example = {
    "input": [[1, 0], [0, 2]],
    "output": [[1, 0], [0, 2]],
}
```

Then `example["input"]` retrieves the input grid, and `example["output"]` retrieves the output grid.

The quotation marks matter: `"input"` and `"output"` are strings used as keys.

In [ ]:
# Run this example.

example = {
    "input": [
        [1, 0, 0],
        [0, 2, 0],
    ],
    "output": [
        [1, 0],
        [0, 2],
        [0, 0],
    ],
}

print("Input grid:", example["input"])
print("Output grid:", example["output"])
show_pair(example["input"], example["output"], "One ARC example")

## Dictionary practice

The dictionary below has already been defined. Complete the two lines by retrieving its values:

- Set `practice_input` to the value stored under `"input"`.
- Set `practice_output` to the value stored under `"output"`.

Do not type the grids again, and do not change `practice_example`. Use square brackets and the appropriate key.

In [ ]:
# Complete these two lines.

practice_example = {
    "input": [[3, 0], [0, 4]],
    "output": [[3, 0], [0, 4]],
}

practice_input = None
practice_output = None

In [ ]:
#@title Check Dictionary Practice

def check_dictionary_practice():
    expected_input = [[3, 0], [0, 4]]
    expected_output = [[3, 0], [0, 4]]
    expected_dictionary = {
        "input": expected_input,
        "output": expected_output,
    }

    if practice_example != expected_dictionary:
        print("The dictionary changed. Restore its original input and output values.")
        return
    if practice_input is not practice_example["input"]:
        print('Retrieve the existing value with practice_example["input"] instead of recreating it.')
        return
    if practice_output is not practice_example["output"]:
        print('Retrieve the existing value with practice_example["output"] instead of recreating it.')
        return
    print("Great work. You retrieved both dictionary values without changing the dictionary.")


make_check_button(check_dictionary_practice)

## Part 1: Study the evidence

An ARC task usually contains:

- several **training pairs**, each showing an input and its correct output
- one or more **test inputs**, whose outputs you must predict

Every training pair follows the same hidden rule. Your program must capture that rule generally—not memorize one picture.

Run the next cell. Before reading further, study what changes and what stays the same.

In [ ]:
training_example_a = {
    "input": [
        [1, 0, 0],
        [0, 2, 0],
    ],
    "output": [
        [1, 0],
        [0, 2],
        [0, 0],
    ],
}

training_example_b = {
    "input": [
        [0, 3],
        [3, 0],
        [0, 4],
    ],
    "output": [
        [0, 3, 0],
        [3, 0, 4],
    ],
}

training_example_c = {
    "input": [
        [5, 0, 6, 0],
        [0, 7, 0, 8],
    ],
    "output": [
        [5, 0],
        [0, 7],
        [6, 0],
        [0, 8],
    ],
}

training_examples = [training_example_a, training_example_b, training_example_c]

show_examples(training_examples)

## Observation notebook

Write your answers before coding. Precise observations are more useful than a quick guess.

1. Do any colors appear or disappear?
2. Does the number of cells change?
3. What happens to the input's width and height?
4. Where does the top row of the input appear in the output?
5. Track one colored cell. If it begins at `(row, column)`, where does it end?
6. State your proposed rule in one sentence.

### My observations

- **Colors:**
- **Number of cells:**
- **Width and height:**
- **A cell's new position:**
- **My proposed rule:**

## Exercise 1: Predict the output size

The examples suggest that the output's width and height trade places.

Write `predicted_output_size(grid)` so it returns the output size as `(width, height)`.

For a `4 × 2` input—width 4 and height 2—the predicted output size is `(2, 4)`.

In [ ]:
# Write your solution for Exercise 1 here

def predicted_output_size(grid):
    return None

In [ ]:
#@title Check Exercise 1

def check_exercise_1():
    tests = [
        ([[1, 2, 3], [4, 5, 6]], (2, 3)),
        ([[1, 2], [3, 4], [5, 6]], (3, 2)),
        ([[7]], (1, 1)),
        ([[1, 2, 3, 4]], (1, 4)),
    ]
    try:
        for grid, expected in tests:
            got = predicted_output_size(grid)
            if got != expected:
                print(f"Close: got {got} for {grid}; expected {expected}.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Great work. Exercise 1 is correct.")


make_check_button(check_exercise_1)

## Reasoning checkpoint: Predict a new output

Before the rule is named, apply your hypothesis to one new input.

Write the complete predicted grid in `prediction_output`. Use a list of lists, just like the other grids. Do not write transformation code yet—this exercise is about making your idea precise enough to test.

In [ ]:
prediction_input = [
    [1, 2, 3],
    [4, 5, 6],
]

show_grid(prediction_input, "Prediction input")

# Replace None with your predicted output grid.
prediction_output = None

In [ ]:
#@title Check Prediction

def check_prediction():
    expected = [
        [1, 4],
        [2, 5],
        [3, 6],
    ]
    if prediction_output != expected:
        print("Not yet. Compare how each input row and column appears in the training outputs.")
        return
    print("Your prediction matches the hidden rule. Now reveal its name below.")


make_check_button(check_prediction)

## Reveal: the transpose rule

The hidden transformation is **transpose**, the same transpose that we worked with in Week 2! Each input row becomes an output column—or equivalently, each input column becomes an output row.

A cell at `input_grid[row][column]` moves to `output_grid[column][row]`.

For a rectangular input, transposing also swaps the dimensions. This is why the non-square training examples were especially useful evidence.

## Exercise 2: Extract one column

A transpose can be built one input column at a time.

Write `get_column(grid, column_index)` so it returns the values in that column from top to bottom.

Example:

```python
grid = [
    [1, 2, 3],
    [4, 5, 6]
]
```

- `get_column(grid, 0)` returns `[1, 4]`
- `get_column(grid, 2)` returns `[3, 6]`

Use a loop to visit each row and collect the value at `column_index`.

In [ ]:
# Write your solution for Exercise 2 here

def get_column(grid, column_index):
    return None

In [ ]:
#@title Check Exercise 2

def check_exercise_2():
    tests = [
        (([[1, 2, 3], [4, 5, 6]], 0), [1, 4]),
        (([[1, 2, 3], [4, 5, 6]], 2), [3, 6]),
        (([[7], [8], [9]], 0), [7, 8, 9]),
        (([[1, 0], [0, 2], [3, 0]], 1), [0, 2, 0]),
    ]
    try:
        for (grid, column_index), expected in tests:
            got = get_column(grid, column_index)
            if got != expected:
                print(f"Close: column {column_index} was {got}; expected {expected}.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Nice work. Exercise 2 is correct.")


make_check_button(check_exercise_2)

## Exercise 3: Build the complete transformation

Write `transpose_grid(grid)`.

For each input column, call `get_column` and append the returned list as a new output row.

Requirements:

- work for square and rectangular grids
- return a new grid
- leave the input unchanged
- use the helper function you just wrote

You may assume every grid has at least one row and every row has the same length.

In [ ]:
# Write your solution for Exercise 3 here

def transpose_grid(grid):
    return None

In [ ]:
#@title Check Exercise 3

def check_exercise_3():
    tests = [
        (
            [[1, 2, 3], [4, 5, 6]],
            [[1, 4], [2, 5], [3, 6]],
        ),
        (
            [[1, 2], [3, 4], [5, 6]],
            [[1, 3, 5], [2, 4, 6]],
        ),
        ([[7]], [[7]]),
        ([[1, 2, 3, 4]], [[1], [2], [3], [4]]),
    ]
    try:
        for grid, expected in tests:
            original = copy.deepcopy(grid)
            got = transpose_grid(grid)
            if got != expected:
                print("Not yet.")
                print("Input:   ", grid)
                print("Got:     ", got)
                print("Expected:", expected)
                return
            if grid != original:
                print("The output is right, but the input changed. Build a new grid.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Excellent. Your transpose works on every test and preserves its input.")


make_check_button(check_exercise_3)

## Exercise 4: Write an ARC solver

ARC solvers are commonly written as a function that accepts one input grid and returns one output grid.

Write `solve(grid)` by calling `transpose_grid(grid)`. Keeping `solve` separate will make it easy to reuse the same testing workflow for different ARC rules later.

In [ ]:
# Write your solution for Exercise 4 here

def solve(grid):
    return None

In [ ]:
#@title Check Exercise 4

def check_exercise_4():
    extra_tests = [
        {
            "input": [[9, 0, 0], [0, 8, 0], [0, 0, 7], [6, 0, 0]],
            "output": [[9, 0, 0, 6], [0, 8, 0, 0], [0, 0, 7, 0]],
        },
        {
            "input": [[1, 2, 3, 4, 5], [6, 7, 8, 9, 0]],
            "output": [[1, 6], [2, 7], [3, 8], [4, 9], [5, 0]],
        },
    ]
    try:
        for example in training_examples + extra_tests:
            input_grid = example["input"]
            expected = example["output"]
            original = copy.deepcopy(input_grid)
            got = solve(input_grid)
            if got != expected:
                print("Your solver does not yet generalize to this grid:")
                print(input_grid)
                return
            if input_grid != original:
                print("Your solver changed its input. Return a new grid instead.")
                return
    except Exception as exc:
        print("Your solver raised an error:", exc)
        return
    print("Success! Your solver passes the training examples and new test cases.")


make_check_button(check_exercise_4)

## Apply the rule to a test input

The final input below was not part of the training set. Run your solver, inspect the picture, and confirm that the same rule still makes sense.

In [ ]:
test_input = [
    [1, 0, 2, 0, 3],
    [0, 4, 0, 5, 0],
    [6, 0, 7, 0, 8],
]

test_output = solve(test_input)
if test_output is None:
    print("Finish solve(grid), then run this cell again.")
else:
    show_pair(test_input, test_output, "Unseen test case")

# Stretch Laboratory

The core lesson used one rule. The challenges below build a small **transformation engine** that can represent, combine, and infer several possible rules.

Complete these in order. They deliberately become much less guided.

## Stretch 1: A family of geometric transformations

Implement all five functions:

- `identity(grid)` returns an unchanged copy
- `flip_horizontal(grid)` reverses every row
- `flip_vertical(grid)` reverses the order of the rows
- `rotate90_clockwise(grid)` rotates 90° clockwise
- `rotate180(grid)` rotates 180°

Try to define the rotations by **composing** transpose and flip functions instead of rebuilding every result cell by cell.

A useful scientific question: in what order should transpose and flip be applied to produce a clockwise rotation?

In [ ]:
# Stretch 1

def identity(grid):
    return None


def flip_horizontal(grid):
    return None


def flip_vertical(grid):
    return None


def rotate90_clockwise(grid):
    return None


def rotate180(grid):
    return None

In [ ]:
#@title Check Stretch 1

def check_stretch_1():
    grid = [[1, 2, 3], [4, 5, 6]]
    original = copy.deepcopy(grid)
    try:
        results = [
            ("identity", identity(grid), [[1, 2, 3], [4, 5, 6]]),
            ("flip_horizontal", flip_horizontal(grid), [[3, 2, 1], [6, 5, 4]]),
            ("flip_vertical", flip_vertical(grid), [[4, 5, 6], [1, 2, 3]]),
            ("rotate90_clockwise", rotate90_clockwise(grid), [[4, 1], [5, 2], [6, 3]]),
            ("rotate180", rotate180(grid), [[6, 5, 4], [3, 2, 1]]),
        ]
        for name, got, expected in results:
            if got != expected:
                print(f"{name} returned {got}; expected {expected}.")
                return
            if grid != original or got is grid:
                print(f"{name} must return a new grid without changing the input.")
                return
    except Exception as exc:
        print("A transformation raised an error:", exc)
        return
    print("Stretch 1 complete: all five transformations work.")


make_check_button(check_stretch_1)

## Stretch 2: A transformation pipeline

Write `apply_steps(grid, steps)`, where `steps` is a list of transformation names.

Apply each named transformation in order and return the final grid.

Examples:

```python
apply_steps(grid, ["transpose", "flip_horizontal"])
apply_steps(grid, ["flip_vertical", "flip_horizontal"])
```

Support these names:

- `"transpose"`
- `"flip_horizontal"`
- `"flip_vertical"`
- `"rotate90"`
- `"rotate180"`

Inside the loop, use `if` / `elif` to decide which function to call. For an unknown name, raise `ValueError("Unknown transformation: " + name)`.

In [ ]:
# Stretch 2

def apply_steps(grid, steps):
    result = identity(grid)
    for name in steps:
        # Use if / elif to choose and apply the named transformation.
        pass
    return result

In [ ]:
#@title Check Stretch 2

def check_stretch_2():
    grid = [[1, 2, 3], [4, 5, 6]]
    tests = [
        ([], [[1, 2, 3], [4, 5, 6]]),
        (["transpose"], [[1, 4], [2, 5], [3, 6]]),
        (["transpose", "flip_horizontal"], [[4, 1], [5, 2], [6, 3]]),
        (["flip_vertical", "flip_horizontal"], [[6, 5, 4], [3, 2, 1]]),
        (["rotate90", "rotate90"], [[6, 5, 4], [3, 2, 1]]),
    ]
    try:
        for steps, expected in tests:
            original = copy.deepcopy(grid)
            got = apply_steps(grid, steps)
            if got != expected:
                print(f"Steps {steps} returned {got}; expected {expected}.")
                return
            if grid != original:
                print("The pipeline changed its original input.")
                return
        try:
            apply_steps(grid, ["teleport"])
            print("Unknown transformation names should raise ValueError.")
            return
        except ValueError:
            pass
    except Exception as exc:
        print("The pipeline raised an unexpected error:", exc)
        return
    print("Stretch 2 complete: your transformation pipeline works.")


make_check_button(check_stretch_2)

## Stretch 3: Let the computer infer the rule

So far, *you* inferred the rule. Now write code that compares several candidate rules against the evidence.

`RULE_NAMES` is a list of names. The provided `apply_rule` helper applies one name by reusing your transformation pipeline.

1. Write `matching_rules(input_grid, output_grid)` to return an alphabetically sorted list of every rule that produces that output.
2. Write `infer_rules(examples)` to return the rules that match **every** example dictionary. Retrieve its grids with `example["input"]` and `example["output"]`.

Do not stop at the first match. One example can be ambiguous: a symmetric grid might look unchanged after several different transformations.

In [ ]:
# Stretch 3

RULE_NAMES = [
    "identity",
    "transpose",
    "flip_horizontal",
    "flip_vertical",
    "rotate90",
    "rotate180",
]


def apply_rule(name, grid):
    if name == "identity":
        return identity(grid)
    return apply_steps(grid, [name])


def matching_rules(input_grid, output_grid):
    return None


def infer_rules(examples):
    return None

In [ ]:
#@title Check Stretch 3

def check_stretch_3():
    try:
        ordinary = [[1, 2, 3], [4, 5, 6]]
        transposed = [[1, 4], [2, 5], [3, 6]]
        got = matching_rules(ordinary, transposed)
        if got != ["transpose"]:
            print("For the rectangular example, expected ['transpose']; got", got)
            return

        symmetric = [[1, 2], [2, 3]]
        got = matching_rules(symmetric, symmetric)
        expected = ["identity", "transpose"]
        if got != expected:
            print("The symmetric example has two matches.")
            print("Expected:", expected)
            print("Got:     ", got)
            return

        ambiguous_first = {
            "input": symmetric,
            "output": symmetric,
        }
        revealing_second = {
            "input": [[1, 0, 0], [0, 2, 3]],
            "output": [[1, 0], [0, 2], [0, 3]],
        }
        if infer_rules([ambiguous_first]) != expected:
            print("Keep every rule consistent with the evidence, not only one.")
            return
        if infer_rules([ambiguous_first, revealing_second]) != ["transpose"]:
            print("The second example should narrow the possibilities to transpose.")
            return
    except Exception as exc:
        print("Rule inference raised an error:", exc)
        return
    print("Stretch 3 complete: your program can infer candidate rules!")


make_check_button(check_stretch_3)

## Stretch 4: Design an experiment that separates two hypotheses

Suppose both `identity` and `transpose` explain the first example. A useful next example is a **counterexample**: an input on which those rules predict different outputs.

Write `find_distinguishing_grid(rule_a, rule_b, candidate_grids)`.

- `rule_a` and `rule_b` are names from `RULE_NAMES`.
- Use `apply_rule` to apply each named rule.
- Return the first candidate grid for which the two transformations produce different results.
- Return `None` if none of the candidates separates them.
- Do not modify the candidate grids.

This is more than a programming trick. Designing examples that distinguish competing explanations is a core part of scientific reasoning.

In [ ]:
# Stretch 4

def find_distinguishing_grid(rule_a, rule_b, candidate_grids):
    return None


candidate_grids = [
    [[1]],
    [[1, 2], [2, 3]],
    [[1, 2, 3], [4, 5, 6]],
]

# After implementing the function, try different pairs of rules.
print(find_distinguishing_grid("identity", "transpose", candidate_grids))

In [ ]:
#@title Check Stretch 4

def check_stretch_4():
    candidates = [
        [[1]],
        [[1, 2], [2, 3]],
        [[1, 2, 3], [4, 5, 6]],
    ]
    original = copy.deepcopy(candidates)
    try:
        got = find_distinguishing_grid("identity", "transpose", candidates)
        expected = [[1, 2, 3], [4, 5, 6]]
        if got != expected:
            print("Expected the first two grids to remain ambiguous and the third to distinguish the rules.")
            print("Got:", got)
            return
        if candidates != original:
            print("The candidate grids were changed.")
            return
        no_difference = find_distinguishing_grid("identity", "identity", candidates)
        if no_difference is not None:
            print("A rule cannot be distinguished from itself; return None.")
            return
    except Exception as exc:
        print("The experiment designer raised an error:", exc)
        return
    print("Stretch 4 complete: you found an input that separates two hypotheses.")


make_check_button(check_stretch_4)

## Stretch 5: Mystery-rule boss battle

The training examples below use one rule from `RULE_NAMES`. This time, no hint names the transformation.

1. Display and inspect the examples.
2. Use `infer_rules` to identify every candidate consistent with the evidence.
3. Choose the supported rule and apply it to `mystery_test_input` with `apply_rule`.
4. Store your final grid in `mystery_prediction`.

Try predicting the output on paper before letting your program produce it.

In [ ]:
mystery_training = [
    {
        "input": [[1, 2, 3], [4, 5, 6]],
        "output": [[4, 1], [5, 2], [6, 3]],
    },
    {
        "input": [[1, 0], [2, 3], [0, 4]],
        "output": [[0, 2, 1], [4, 3, 0]],
    },
]

mystery_test_input = [
    [1, 0, 0, 2],
    [0, 3, 4, 0],
    [5, 0, 0, 6],
]

show_examples(mystery_training)
show_grid(mystery_test_input, "Mystery test input")

In [ ]:
# Stretch 5: complete these two values

mystery_rule_candidates = None
mystery_prediction = None

# When your prediction is ready, display it.
if mystery_prediction is not None:
    show_pair(mystery_test_input, mystery_prediction, "Mystery prediction")

In [ ]:
#@title Check Stretch 5

def check_stretch_5():
    expected_rules = ["rotate90"]
    expected_grid = [
        [5, 0, 1],
        [0, 3, 0],
        [0, 4, 0],
        [6, 0, 2],
    ]
    if mystery_rule_candidates != expected_rules:
        print("Recheck which rules satisfy every training pair.")
        print("Your candidates:", mystery_rule_candidates)
        return
    if mystery_prediction != expected_grid:
        print("You found the rule, but recheck its application to the test input.")
        return
    print("Boss battle complete! You inferred and applied the hidden rule.")


make_check_button(check_stretch_5)

## Stretch 6: Become the puzzle designer

Create your own mini ARC task.

Requirements:

1. Choose one rule—or a pipeline of two rules.
2. Create at least **three** training example dictionaries of different shapes or sizes. Give each one an `"input"` and an `"output"`.
3. Include one test input.
4. Do not write the rule beside the examples.
5. Ask another person to infer it.
6. Use your code to verify every expected output.

Then make the task harder in a meaningful way. Possible experiments:

- Make the first example ambiguous but let later examples resolve the ambiguity.
- Use repeated colors so individual cells are harder to track.
- Compare square examples with rectangular examples.
- Build a two-step rule and investigate whether another shorter rule produces the same outputs.
- Write a function that generates random candidate grids and searches for one that distinguishes two rules.

Use as many new code and text cells as you need below this one.

In [ ]:
# Stretch 6 workspace

my_training_examples = []
my_test_input = None
my_expected_test_output = None

# Add examples above, then uncomment these lines to display your task.
# show_examples(my_training_examples)
# show_grid(my_test_input, "My test input")

## Wrap-up and reflection

You followed an ARC-style reasoning loop:

1. **Observe** several examples.
2. **Hypothesize** a rule.
3. **Implement** the rule as a general function.
4. **Test** it on every training pair.
5. **Generalize** it to a new input.
6. **Revise** when new evidence contradicts the hypothesis.

Answer these questions:

- What evidence convinced you that the core rule was transpose?
- Why were rectangular grids more informative than some square grids?
- Which bug or incorrect hypothesis taught you the most?
- In Stretch 3, why was it important to return every matching rule?
- What makes a training example useful for distinguishing two rules?
- Which stretch challenge would you like to extend next?

### My reflection

- **Most convincing evidence:**
- **Why rectangular grids helped:**
- **Most useful mistake:**
- **Why multiple matching rules matter:**
- **What makes an example informative:**
- **What I want to explore next:**

## Share your completed notebook

When you finish:

1. Confirm that all four core checks report success.
2. Complete as many stretch challenges as time allows.
3. In Colab, click **Share**.
4. Add **michael.d.moffitt@gmail.com**.
5. Set permission to **Viewer** or **Commenter**.
6. Send the share invite.

Include your invented puzzle and any questions that came up. A hypothesis that fails on a new example is not wasted work—it is evidence that helps you build a better rule.